# visualizeLOB 演示

本 Notebook 演示订单簿(Limit Order Book)动态可视化工具的完整工作流:

1. **生成 toy 数据** —— 调用 `generate_toy_data()` 生成逻辑自洽的订单簿数据
2. **读入数据** —— 用 `LOBDataLoader` 读取 parquet 并按需过滤
3. **静态单帧** —— 用 `LOBVisualizer.plot_single_frame()` 查看某一帧及其与上一帧的差异
4. **动态回放** —— 用 `LOBVisualizer.plot_animation()` 逐帧播放订单簿的演化

约定: **蓝色 = 买盘, 红色 = 卖盘**; **深色 = 挂单量增加, 浅色 = 挂单量减少(撤单/成交)**。

## 1. 生成 toy 数据

`generate_toy_data()` 内部用一个真实的撮合引擎(`InternalOrderBook`)模拟挂单、撤单、成交,
因此产出的数据严格符合订单簿的工作原理(不会出现跳档成交、相邻两帧多个价位同时增量等不可能的情况)。

产物: `toy_data/orderbook.parquet`(101 帧快照) 与 `toy_data/triggerInfo.parquet`(100 个触发事件)。

In [ ]:
# 导入工具模块
import visualize_lob as vl

# 生成 toy 数据(写入 toy_data/ 目录), 返回两个 DataFrame 便于预览
orderbook_df, trigger_df = vl.generate_toy_data(out_dir='toy_data', n_events=100, seed=42)

# 预览 orderbook 的前几行(只看前 3 档以保持简洁)
orderbook_df[['code', 'adjIndex', 'time',
              'bidPx1', 'bidVlm1', 'askPx1', 'askVlm1',
              'bidPx2', 'bidVlm2', 'askPx2', 'askVlm2']].head()

In [ ]:
# 预览 triggerInfo: 每一行是导致对应 adjIndex 那帧变化的行情类型
trigger_df.head()

## 2. 读入数据

`LOBDataLoader` 读取两个 parquet 文件,并支持按 **股票代码 / 时间区间 / adjIndex 区间** 过滤。
`(code, adjIndex)` 双元组是连接 orderbook 与 triggerInfo 两张表的主键。

> 注:`time` 与 `serverTime` 为 `HHMMSSmmm` 格式的整数(时×10⁷ + 分×10⁵ + 秒×10³ + 毫秒),
> 例如 `93949000` 表示 09:39:49.000。下方预览中这两列即以该整数形式显示。

In [ ]:
# 创建数据加载器
loader = vl.LOBDataLoader('toy_data/orderbook.parquet', 'toy_data/triggerInfo.parquet')
print('总帧数:', len(loader))

# 查看第 0 帧(初始帧, 无触发事件)
frame0 = loader.get_frame(0)
print('第 0 帧 code =', int(frame0['code']), ', adjIndex =', int(frame0['adjIndex']))
print('第 0 帧触发信息:', loader.get_trigger(0))

# 查看第 8 帧的触发信息
print('第 8 帧触发信息:', dict(loader.get_trigger(8)))

## 3. 静态单帧可视化

`plot_single_frame(pos)` 把第 `pos` 帧画成柱状图: **x 轴是价格, y 轴是挂单量**。
每根柱子从下到上依次为 **留存量(中色) / 新增量(深色) / 减少量残影(浅色)**,
留存量与变化量之间用 **黑色分割线** 区分。

In [ ]:
# 创建可视化器
viz = vl.LOBVisualizer(loader)

# 第 0 帧: 初始状态, 没有与上一帧的差异
viz.plot_single_frame(0)

In [ ]:
# 第 8 帧: 可以看到相对上一帧的变化(深色=新增, 浅色=减少, 黑线=分割线)
viz.plot_single_frame(8)

## 4. 动态回放

`plot_animation()` 生成带 **播放/暂停按钮** 与 **时间滑块** 的交互动画。

动画采用 **两阶段** 方式: 对相邻两帧,先在原位置展示挂单量的增减变化(变化阶段),
再把 x 轴平移到新的盘口位置(平移阶段)。每帧顶部的文字标注会显示该帧的 `triggerType`。

In [ ]:
# 回放全部 101 帧
viz.plot_animation()

## 5. 过滤后回放

也可以先用 `filter()` 框定一个 adjIndex 区间(或时间区间、股票代码),再只回放这一段。

In [ ]:
# 取第 10~30 帧对应的 adjIndex 区间, 过滤后只回放这一段
start_idx = int(loader.get_frame(10)['adjIndex'])
end_idx = int(loader.get_frame(30)['adjIndex'])
loader.filter(start_index=start_idx, end_index=end_idx)
print('过滤后帧数:', len(loader))

# 过滤后重新构造可视化器并回放
viz_sub = vl.LOBVisualizer(loader)
viz_sub.plot_animation()